# Majority-class fallback for all-empty columns
Fills every `Not Applicable` column with the most common training value for that PXD.
Under the new penalizing metric, even a naive majority-class prediction beats `Not Applicable`.

In [1]:
from __future__ import annotations
from pathlib import Path
import pandas as pd

ROOT       = Path('/kaggle/working') if Path('/kaggle').exists() else Path.cwd().parent
TRAIN_PATH = ROOT / 'outputs' / 'train_preprocessed.csv'
SUB_PATH   = ROOT / 'outputs' / 'submission_tier1.csv'
TEST_PATH  = ROOT / 'outputs' / 'test_preprocessed.csv'
OUT_PATH   = ROOT / 'outputs' / 'submission_with_fallback.csv'
NOT_APPLICABLE = 'Not Applicable'

train = pd.read_csv(TRAIN_PATH, dtype=str).fillna(NOT_APPLICABLE)
sub   = pd.read_csv(SUB_PATH,   dtype=str).fillna(NOT_APPLICABLE)
test  = pd.read_csv(TEST_PATH,  dtype=str)
print(f'Train: {train.shape}  Sub: {sub.shape}  Test: {test.shape}')

Train: (36280, 19)  Sub: (1659, 81)  Test: (1659, 3)


## Find empty columns and build majority-class lookup

In [2]:
meta_cols  = [c for c in sub.columns if c not in ('ID','PXD','Raw Data File')]
empty_cols = [c for c in meta_cols if (sub[c] == NOT_APPLICABLE).all()]
filled_cols = [c for c in meta_cols if c not in empty_cols]

print(f'Total metadata columns : {len(meta_cols)}')
print(f'Already filled         : {len(filled_cols)}')
print(f'All-NA (need fallback) : {len(empty_cols)}')
print()
print('Already filled:', filled_cols)

Total metadata columns : 78
Already filled         : 7
All-NA (need fallback) : 71

Already filled: ['Characteristics[BiologicalReplicate]', 'Characteristics[Organism]', 'Characteristics[Sex]', 'Comment[FragmentMassTolerance]', 'Comment[FragmentationMethod]', 'Comment[Instrument]', 'Comment[PrecursorMassTolerance]']


In [3]:
# Per-PXD majority lookup
pxd_majority: dict[str, dict[str, str]] = {}
global_majority: dict[str, str] = {}

for col in empty_cols:
    if col not in train.columns:
        global_majority[col] = NOT_APPLICABLE
        pxd_majority[col] = {}
        continue
    non_na = train[col][train[col] != NOT_APPLICABLE]
    if len(non_na) == 0:
        global_majority[col] = NOT_APPLICABLE
        pxd_majority[col] = {}
        continue
    global_majority[col] = non_na.value_counts().index[0]
    pxd_majority[col] = {}
    for pxd, grp in train.groupby('PXD'):
        vals = grp[col][grp[col] != NOT_APPLICABLE]
        if len(vals) > 0:
            pxd_majority[col][pxd] = vals.value_counts().index[0]

print(f'Lookup built for {len(pxd_majority)} columns.')

Lookup built for 71 columns.


## Apply fallback and save

In [4]:
sub_out = sub.copy()
if 'PXD' not in sub_out.columns:
    sub_out['PXD'] = test['PXD'].values

results_log = []

for col in empty_cols:
    pxd_map    = pxd_majority.get(col, {})
    global_val = global_majority.get(col, NOT_APPLICABLE)
    sub_out[col] = sub_out['PXD'].apply(lambda pxd: pxd_map.get(pxd, global_val))
    n_filled = (sub_out[col] != NOT_APPLICABLE).sum()
    results_log.append({'column': col, 'n_filled': n_filled, 'global_val': global_val})

sub_out = sub_out[sub.columns.tolist()]  # restore original column order
sub_out.to_csv(OUT_PATH, index=False)
print(f'Saved → {OUT_PATH}')
print(f'Shape : {sub_out.shape}')

Saved → c:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\outputs\submission_with_fallback.csv
Shape : (1659, 81)


## Summary — how many rows were filled per column

In [5]:
import pandas as pd
log_df = pd.DataFrame(results_log).sort_values('n_filled', ascending=False)
print('Top 20 columns by rows filled:')
display(log_df.head(20))

still_empty = log_df[log_df['n_filled'] == 0]['column'].tolist()
print(f'\nColumns with zero training signal: {len(still_empty)}')
for c in still_empty:
    print(f'  {c}')

print('\nUpload outputs/submission_with_fallback.csv to Kaggle.')

Top 20 columns by rows filled:


,column,n_filled,global_val
0,Characteristics[Age],1659,not available
6,Characteristics[CellLine],1659,not available
3,Characteristics[AncestryCategory],1659,not available
15,Characteristics[Disease],1659,not available
21,Characteristics[Modification],1659,NT=Carbamidomethyl;AC=UNIMOD:4;TA=C;MT=Fixed;N...
13,Characteristics[DevelopmentalStage],1659,not available
8,Characteristics[CellType],1659,not available
31,Characteristics[OrganismPart],1659,blood plasma
53,Comment[FractionIdentifier],1659,1
1,Characteristics[AlkylationReagent],0,Not Applicable



Columns with zero training signal: 62
  Characteristics[AlkylationReagent]
  Characteristics[AnatomicSiteTumor]
  Characteristics[Bait]
  Characteristics[BMI]
  Characteristics[CleavageAgent]
  Characteristics[Compound]
  Characteristics[DiseaseTreatment]
  Characteristics[Depletion]
  Characteristics[Genotype]
  Characteristics[GeneticModification]
  Characteristics[GrowthRate]
  Characteristics[Label]
  Characteristics[MaterialType]
  Characteristics[ConcentrationOfCompound]
  Characteristics[Modification].1
  Characteristics[Modification].2
  Characteristics[Modification].4
  Characteristics[Modification].3
  Characteristics[Modification].6
  Characteristics[NumberOfBiologicalReplicates]
  Characteristics[NumberOfSamples]
  Characteristics[Modification].5
  Characteristics[NumberOfTechnicalReplicates]
  Characteristics[OriginSiteDisease]
  Characteristics[PooledSample]
  Characteristics[ReductionReagent]
  Characteristics[SamplingTime]
  Characteristics[Specimen]
  Characteristics[